In [269]:
!pip install gensim

In [270]:
import gensim
import pandas as pd
import numpy as np

In [271]:
# import gensim.downloader as api

# wv = api.load('word2vec-google-news-300')

In [272]:
wv['king'].shape

(300,)

In [273]:
messages = pd.read_csv('/content/SMSSpamCollection.txt', sep='\t', names=['label', 'message'])
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [274]:
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import re

cached_stopwords = set(stopwords.words())
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [275]:
def process_text(message):
    review = re.sub('[^a-zA-Z]', ' ', message)
    review = review.lower().split()
    review = [lemmatizer.lemmatize(word) for word in review] # if word not in cached_stopwords]
    return ' '.join(review)

In [276]:
 nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [277]:
corpus = messages['message'].apply(process_text)
corpus[:5]

,message
0,go until jurong point crazy available only in ...
1,ok lar joking wif u oni
2,free entry in a wkly comp to win fa cup final ...
3,u dun say so early hor u c already then say
4,nah i don t think he go to usf he life around ...


In [278]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

### Gensim `simple_preprocess` Quick Reference
* **Action:** Lowercases, removes punctuation, and splits text into tokens.
* **Defaults:** Keeps words with lengths between 2 and 15 characters.
* **Accents:** Set `deacc=True` to strip accent marks.

In [279]:
words = []

def process_tokenize_simple_preprocess(message):
    message= sent_tokenize(message)
    for sent in message:
        words.append(simple_preprocess(sent))

In [280]:
# nltk.download('punkt_tab')
sent_tokenize('hi i am aum. hi. hello')

['hi i am aum.', 'hi.', 'hello']

In [281]:
corpus.apply(process_tokenize_simple_preprocess)
len(words)

5569

In [282]:
sent_tokenize(corpus[0])

['go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat']

In [283]:
words[0]

['go',
 'until',
 'jurong',
 'point',
 'crazy',
 'available',
 'only',
 'in',
 'bugis',
 'great',
 'world',
 'la',
 'buffet',
 'cine',
 'there',
 'got',
 'amore',
 'wat']

In [284]:
model = gensim.models.Word2Vec(words)
model

In [285]:
model.wv.index_to_key[:10]

['to', 'you', 'the', 'and', 'it', 'in', 'is', 'me', 'my', 'for']

In [286]:
model.corpus_count, model.total_train_time, model.corpus_total_words, model.epochs

(5569, 0.5329364049985088, 78225, 5)

In [287]:
model.wv.similar_by_word('kid')

[('even', 0.9974855184555054),
 ('into', 0.9974590539932251),
 ('were', 0.9973123669624329),
 ('smile', 0.9972708821296692),
 ('over', 0.9972667098045349),
 ('without', 0.9972352981567383),
 ('never', 0.997161865234375),
 ('fine', 0.997154951095581),
 ('down', 0.9971516132354736),
 ('lot', 0.9971369504928589)]

In [288]:
model.wv.similar_by_word('send')

[('msg', 0.9987635612487793),
 ('st', 0.9983583092689514),
 ('entry', 0.9983310103416443),
 ('week', 0.9982621669769287),
 ('poly', 0.9981511831283569),
 ('wk', 0.998143196105957),
 ('sm', 0.998124361038208),
 ('service', 0.9981160163879395),
 ('win', 0.9980884194374084),
 ('com', 0.9980684518814087)]

In [289]:
model.wv['king'].shape

(100,)

In [290]:
words[0]

['go',
 'until',
 'jurong',
 'point',
 'crazy',
 'available',
 'only',
 'in',
 'bugis',
 'great',
 'world',
 'la',
 'buffet',
 'cine',
 'there',
 'got',
 'amore',
 'wat']

## AvgWord2Vec

so as we can see above that there are currently some words above for each one of them there will be a vector created of shape (100,) but what we want is that we get `one vector` with shape (100,)

so what avgword2vec does is that it creates `average` of all the vectors in the senctece

In [291]:
def avg_word2vec(doc):
    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key], axis=0)

In [292]:
!pip install tqdm

In [293]:
from tqdm import tqdm

In [294]:
X = []

for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

  6%|▌         | 320/5569 [00:00<00:03, 1635.41it/s]/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 5569/5569 [00:01<00:00, 4463.77it/s]


In [295]:
len(X), X[1].shape

(5569, (100,))

In [296]:
for a in X:
    if a.shape != (100,):
        print(a.shape)

()
()
()
()
()
()
()
()
()
()
()
()


In [297]:
X[0]

array([-0.21357144,  0.26287538,  0.05204296,  0.08814629,  0.09667806,
       -0.5238274 ,  0.12769945,  0.4964904 , -0.29780585, -0.1010222 ,
       -0.19853878, -0.4346363 , -0.0338783 ,  0.05214487,  0.24188294,
       -0.15711544,  0.08745887, -0.29874173, -0.04888177, -0.5558633 ,
        0.1939837 ,  0.17889637,  0.09758995, -0.19999345, -0.12500492,
        0.00732258, -0.3738645 , -0.25373524, -0.27048916,  0.02788701,
        0.30232468, -0.01830669,  0.06980088, -0.24038503, -0.1344142 ,
        0.4089538 ,  0.01108928, -0.16863284, -0.12461995, -0.3876783 ,
        0.14285555, -0.2654397 , -0.2046034 , -0.01604822,  0.25590995,
       -0.0162362 , -0.18617862, -0.06010391,  0.08521939,  0.13989654,
        0.15059684, -0.18002278, -0.03012684,  0.09597065, -0.11140104,
        0.21637596,  0.17662913,  0.05666507, -0.30295026,  0.13420576,
        0.00423795,  0.12425001, -0.06866232, -0.1563553 , -0.31322643,
        0.23523577,  0.0637384 ,  0.15083504, -0.37075272,  0.32

In [298]:
len(X)

5569

In [299]:
X_clean = [a if isinstance(a, np.ndarray) and a.shape == (100,) else np.zeros(100) for a in X]
X_new = np.array(X_clean)
X_new.shape

(5569, 100)

In [300]:
messages.shape

(5572, 2)

## the 3 entries were redundant like (-:, etc

In [301]:
[[i, j, k] for i, j, k in zip(list(map(len, corpus)), corpus, messages['message']) if i < 1]

[[0, '', '645'], [0, '', ':) '], [0, '', ':-) :-)']]

In [302]:
y = messages[list(map(lambda x: len(x) > 0, corpus))]
y = pd.get_dummies(y['label'])
y = y.iloc[:,0].values
y.shape

(5569,)

In [303]:
df = [pd.DataFrame(X[i].reshape(1, -1)) for i in range(len(X))]
df = pd.concat(df, ignore_index=True)
df.shape

/tmp/ipykernel_553/252073354.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(df, ignore_index=True)


(5569, 100)

In [304]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.213571,0.262875,0.052043,0.088146,0.096678,-0.523827,0.127699,0.496490,-0.297806,-0.101022,...,0.292535,0.146731,0.051784,0.127135,0.345413,0.307961,0.109951,-0.194451,0.177465,-0.127869
1,-0.180829,0.222603,0.041668,0.079792,0.081089,-0.455442,0.098959,0.429364,-0.247670,-0.070886,...,0.249745,0.120701,0.039063,0.106569,0.289902,0.255363,0.090165,-0.174307,0.145205,-0.106028
2,-0.209687,0.286599,0.072778,0.116985,0.100127,-0.580595,0.128914,0.504296,-0.313976,-0.119410,...,0.292841,0.159764,0.041258,0.129248,0.370436,0.296751,0.085840,-0.228331,0.212499,-0.139336
3,-0.289633,0.350916,0.063255,0.118209,0.128857,-0.692718,0.165606,0.669514,-0.402361,-0.121795,...,0.390441,0.188150,0.060890,0.173255,0.446066,0.413901,0.147768,-0.269344,0.228207,-0.156043
4,-0.251553,0.298921,0.061970,0.096111,0.120516,-0.593823,0.140143,0.575235,-0.343087,-0.115843,...,0.340394,0.159696,0.056900,0.148508,0.385697,0.355879,0.125616,-0.236019,0.198548,-0.145068


In [305]:
df.isna().sum()

,0
0,12
1,12
2,12
3,12
4,12
...,...
95,12
96,12
97,12
98,12


In [306]:
df.shape, y.shape

((5569, 100), (5569,))

In [307]:
df['Ouput'] = y

In [308]:
X = df.dropna(axis=0)
X

,0,1,2,3,4,5,6,7,8,9,...,91,92,93,94,95,96,97,98,99,Ouput
0,-0.213571,0.262875,0.052043,0.088146,0.096678,-0.523827,0.127699,0.496490,-0.297806,-0.101022,...,0.146731,0.051784,0.127135,0.345413,0.307961,0.109951,-0.194451,0.177465,-0.127869,True
1,-0.180829,0.222603,0.041668,0.079792,0.081089,-0.455442,0.098959,0.429364,-0.247670,-0.070886,...,0.120701,0.039063,0.106569,0.289902,0.255363,0.090165,-0.174307,0.145205,-0.106028,True
2,-0.209687,0.286599,0.072778,0.116985,0.100127,-0.580595,0.128914,0.504296,-0.313976,-0.119410,...,0.159764,0.041258,0.129248,0.370436,0.296751,0.085840,-0.228331,0.212499,-0.139336,False
3,-0.289633,0.350916,0.063255,0.118209,0.128857,-0.692718,0.165606,0.669514,-0.402361,-0.121795,...,0.188150,0.060890,0.173255,0.446066,0.413901,0.147768,-0.269344,0.228207,-0.156043,True
4,-0.251553,0.298921,0.061970,0.096111,0.120516,-0.593823,0.140143,0.575235,-0.343087,-0.115843,...,0.159696,0.056900,0.148508,0.385697,0.355879,0.125616,-0.236019,0.198548,-0.145068,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5564,-0.252014,0.330029,0.082957,0.123859,0.120061,-0.654748,0.158717,0.586940,-0.361418,-0.138603,...,0.176518,0.053967,0.136365,0.425434,0.349695,0.113245,-0.250786,0.236836,-0.153097,False
5565,-0.267261,0.307904,0.070220,0.116370,0.134648,-0.638612,0.143235,0.603976,-0.366168,-0.117452,...,0.176594,0.045188,0.158634,0.418799,0.373654,0.118429,-0.258691,0.216735,-0.148380,True
5566,-0.291711,0.360993,0.067613,0.111148,0.122724,-0.696948,0.170789,0.668996,-0.398543,-0.138037,...,0.200027,0.073880,0.173413,0.463047,0.418060,0.153844,-0.252154,0.239929,-0.173542,True
5567,-0.263406,0.325051,0.066905,0.106634,0.118005,-0.638632,0.150515,0.605476,-0.362656,-0.128089,...,0.177412,0.070298,0.153710,0.423216,0.375778,0.128866,-0.239326,0.218634,-0.159593,True


In [309]:
df['Ouput'].shape

(5569,)

In [310]:
X, y = X, X['Ouput']
X.shape, y.shape

((5557, 101), (5557,))

In [311]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X.drop(['Ouput'], axis=1), y, test_size=0.2, random_state=0)

In [312]:
X_train.shape, y_train.shape

((4445, 100), (4445,))

In [313]:
from sklearn.ensemble import RandomForestClassifier

In [314]:
model = RandomForestClassifier()
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.9712230215827338

In [315]:
from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

       False       0.94      0.85      0.89       157
        True       0.98      0.99      0.98       955

    accuracy                           0.97      1112
   macro avg       0.96      0.92      0.94      1112
weighted avg       0.97      0.97      0.97      1112



## Prcoess

  1. clean data:
      - remove things besides a-zA-z and lower it
      - remove stopwords
      - do lemmatization
  2. train word2vec using gensim for `words`
  3. In mid some data which was bad was cleaned
  4. learn about `simple_preprocess`
  5. do `avg_word2vec` to make one vector out of whole sentence